# DocFinder — Pipeline Colab v2 (BGE-M3)

Ce notebook tourne deux choses en parallèle sur Colab GPU :

1. Le serveur FastAPI `/encode` (port 8001) → Mac appelle pour encoder les requêtes BGE-M3.
2. Le pipeline d'indexation (BGE-M3 → Qdrant via le Mac).

**Les deux partagent le même modèle en VRAM** (un seul chargement GPU, ≈ 2.3 GB fp16).

## Pré-requis

- Runtime GPU activé (Exécution → Modifier le type d'exécution → T4)
- Tunnel Cloudflare #1 lancé côté Mac (expose `localhost:8000` → `docfinder.jinkohub.digital`)
- Tunnel Cloudflare #2 configuré (expose `localhost:8001` Colab → `encode.jinkohub.digital`) — token fourni à la cellule 3
- `COLAB_QUERY_TOKEN` généré côté Mac (même valeur des deux côtés)

## 1. Cloner / mettre à jour le repo

In [ ]:
import os, subprocess
BRANCH = 'claude/review-project-cloudflare-ggcYD'
REPO = 'https://github.com/kofekod23/docfinder.git'
if not os.path.exists('/content/docfinder/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/content/docfinder'], check=True)
else:
    subprocess.run(['git', '-C', '/content/docfinder', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/docfinder', 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/docfinder', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
print('HEAD →', subprocess.check_output(['git', '-C', '/content/docfinder', 'rev-parse', '--short', 'HEAD']).decode().strip())

## 2. Installer les dépendances

⚠️ Après cette cellule : **Exécution → Redémarrer la session** puis relance à partir de la cellule 3.

In [ ]:
!pip install -q FlagEmbedding==1.3.5 'transformers>=4.45,<4.50' 'fastapi>=0.110' 'uvicorn[standard]>=0.27' httpx pydantic
print('\n⚠️  Redémarre le runtime maintenant (Exécution → Redémarrer la session) puis relance cellule 3.')

## 3. Installer cloudflared et lancer le tunnel #2

Ce tunnel expose `localhost:8001` (le serveur `/encode`) vers `encode.jinkohub.digital`.
Colle le token du tunnel #2 dans la variable `TUNNEL2_TOKEN` ci-dessous.

In [ ]:
TUNNEL2_TOKEN = ''  # ← colle ici le token du 2e tunnel (cloudflared tunnel run --token ...)

import subprocess, os, time
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-qO', '/usr/local/bin/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

assert TUNNEL2_TOKEN.strip(), 'TUNNEL2_TOKEN vide — colle le token du 2e tunnel'
log = open('/content/cloudflared.log', 'w')
proc = subprocess.Popen(['cloudflared', 'tunnel', 'run', '--token', TUNNEL2_TOKEN],
                        stdout=log, stderr=subprocess.STDOUT)
time.sleep(5)
print('cloudflared PID', proc.pid, '— logs: /content/cloudflared.log')

## 4. Variables d'environnement

À renseigner (mêmes valeurs que le `.env` du Mac) :

- `COLAB_QUERY_TOKEN` — secret partagé pour `/encode`.
- `CF_ACCESS_CLIENT_ID` / `CF_ACCESS_CLIENT_SECRET` — service token Cloudflare Access (sinon les requêtes Colab → Mac reçoivent un HTML de login et `r.json()` crashe).

In [ ]:
import os
os.environ['MAC_BASE_URL']            = 'https://docfinder.jinkohub.digital'
os.environ['DOCFINDER_ROOT']          = '/Users/julien/Documents'
os.environ['COLAB_QUERY_TOKEN']       = ''  # ← même valeur que Mac .env
os.environ['CF_ACCESS_CLIENT_ID']     = ''  # ← Cloudflare Access service token ID (même valeur que Mac .env)
os.environ['CF_ACCESS_CLIENT_SECRET'] = ''  # ← Cloudflare Access service token secret
assert os.environ['COLAB_QUERY_TOKEN'].strip(),       'COLAB_QUERY_TOKEN vide'
assert os.environ['CF_ACCESS_CLIENT_ID'].strip(),     'CF_ACCESS_CLIENT_ID vide (sinon tunnel Mac renvoie HTML de login)'
assert os.environ['CF_ACCESS_CLIENT_SECRET'].strip(), 'CF_ACCESS_CLIENT_SECRET vide'

## 5. Lancer le serveur `/encode` + le pipeline d'indexation

Cette cellule est **bloquante** : uvicorn tourne en thread daemon, le pipeline occupe le main thread. Interrompre la cellule arrête les deux.

In [ ]:
import sys, os, tempfile, threading
from pathlib import Path
sys.path.insert(0, '/content/docfinder')

import uvicorn
from colab import query_server
from colab.client import MacClient
from colab.embedder_v2 import BGEM3Wrapper
from colab.extractor import extract_text
from colab.pipeline import run_pipeline

mac = MacClient(os.environ['MAC_BASE_URL'])
embedder = BGEM3Wrapper()
embedder._model_or_build()  # force le load GPU une seule fois
tokenizer_decode = lambda ids: embedder._model_or_build().tokenizer.decode(ids)

query_server.set_wrapper(embedder)  # partage du modèle → 1 seul load VRAM

config = uvicorn.Config(query_server.app, host='0.0.0.0', port=8001,
                        log_level='info', access_log=False)
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True, name='query-server').start()
print('query_server OK sur :8001 (tunnel #2 → encode.jinkohub.digital)')

def extractor(path, mode):
    return extract_text(path, mode=mode)

# Colab a déjà une event loop → top-level await (pas asyncio.run())
await run_pipeline(
    os.environ['MAC_BASE_URL'],
    os.environ['DOCFINDER_ROOT'],
    mac_client=mac, extractor=extractor, embedder=embedder,
    tokenizer_decode=tokenizer_decode,
    tmp_dir=Path(tempfile.mkdtemp()),
    checkpoint_path=Path('/content/checkpoint_v2.json'),
)

## 6. Smoke test (optionnel — depuis une autre session)

```bash
curl -s https://encode.jinkohub.digital/healthz
# → {"status":"ok","model":"bge-m3"}

curl -s -X POST https://encode.jinkohub.digital/encode \
  -H 'Content-Type: application/json' \
  -H "X-Auth-Token: $COLAB_QUERY_TOKEN" \
  -d '{"queries":["bonjour"]}' | jq '.dense[0] | length'
# → 1024
```